# RNN-T (transducer) trained from scratch for Icelandic

Course baseline family **Transducer (RNN-T)**. The only listed transducer checkpoint, `nvidia/parakeet-tdt-0.6b-v3`, does not support Icelandic, so we implement and train one on the frozen v1 **train** split (15 h).

- Log-mel features → 4× conv subsampling → 8-layer Transformer encoder; LSTM prediction network; joiner over the train-only character vocabulary.
- Loss: `torchaudio.functional.rnnt_loss`. Decoding: batched greedy search, no language model.
- **Dev** WER selects the checkpoint; **test** is decoded once at the end. All settings: `Config` in `training/train_rnnt.py`, seed 2105.

The notebook stops early and cheaply if something is wrong: unit tests (including torchaudio's loss against a reference implementation), then a 32-clip **memorization check** that must reach a low loss, then a short smoke run, and only then full training.
Kaggle: Accelerator **GPU T4 x2** (one is used), Internet **On**, then **Save Version → Save & Run All (Commit)**. Colab: runtime **T4 GPU**, then **Run all**; keep the tab open. The reported run used Colab (one Tesla T4).

In [ ]:
!nvidia-smi

## 1. Repository at fixed data and code commits

In [ ]:
import os
WORK = "/kaggle/working" if "KAGGLE_KERNEL_RUN_TYPE" in os.environ else "/content"   # Colab also has a /kaggle folder
REPO_DIR = f"{WORK}/icelandic-asr-project3"

DATA_REPO = "https://github.com/RanimAlyaseen/icelandic-asr-project3.git"
DATA_COMMIT = "ef4d5156cb599348dd9dc55934ff391cc8c5ee5d"   # frozen v1 audio, manifests and references
CODE_REPO = "https://github.com/MuhammetGoksenin/icelandic-asr-project3.git"
CODE_COMMIT = "6f702ce9160c443ef18f6b378453fafd743bb50a"
CODE_PATHS = "baselines training tests/test_rnnt.py requirements-modeling.txt"
assert len(CODE_COMMIT) == 40, "pin CODE_COMMIT to a full commit hash"

!which git-lfs >/dev/null || (apt-get -qq update && apt-get -qq install -y git-lfs)
!git lfs install
if not os.path.exists(REPO_DIR):
    !GIT_LFS_SKIP_SMUDGE=1 git clone -q {DATA_REPO} {REPO_DIR}
%cd {REPO_DIR}
!GIT_LFS_SKIP_SMUDGE=1 git checkout -q {DATA_COMMIT}
!git lfs pull --include="data/cleaned/samromur-21.05-20h-v1/**" --exclude=""
!GIT_LFS_SKIP_SMUDGE=1 git fetch -q {CODE_REPO} {CODE_COMMIT}
!git checkout {CODE_COMMIT} -- {CODE_PATHS}
!echo "Data commit: $(git rev-parse HEAD)"; echo "Code commit: {CODE_COMMIT}"
!du -sh data/cleaned/samromur-21.05-20h-v1/*

## 2. Pinned packages and tests

In [ ]:
!pip install -q -r requirements-modeling.txt
!python -m unittest discover -s tests -p "test_rnnt.py" -v 2>&1 | tail -15

## 3. Memorization check (32 clips, ~3 min). Stops the notebook if the loss does not fall.

In [ ]:
import subprocess, re
out = subprocess.run(["python", "training/train_rnnt.py", "--overfit", "32", "--overfit-steps", "600"],
                     capture_output=True, text=True)
print(out.stdout[-4000:], out.stderr[-2000:])
losses = [float(m) for m in re.findall(r"loss\s+([0-9.]+)", out.stdout)]
assert out.returncode == 0 and losses and losses[-1] < 0.1 * losses[0], "memorization failed: stop and inspect"

## 4. Smoke run (256 clips, 2 epochs)

In [ ]:
!python training/train_rnnt.py --train --limit 256 --epochs 2
!rm -rf models/rnnt results/rnnt

## 5. Full training (~2–4 h on one T4)

In [ ]:
!python training/train_rnnt.py --train

In [ ]:
import json
from IPython.display import Image, display
print(json.dumps(json.load(open("results/rnnt/training_summary.json")), indent=2)[:2000])
display(Image("results/rnnt/training_curves.png"))

## 6. Test — once, with the dev-selected checkpoint

In [ ]:
!python training/train_rnnt.py --evaluate-test

## 7. Save results (commit `results/rnnt/` from your own account; publish `models/rnnt/best.pt` on the Hugging Face Hub)

In [ ]:
!pip freeze > results/rnnt/pip_freeze.txt
!cp output/ctc_vocab.json models/rnnt/vocab.json
!cd results && zip -qr {WORK}/rnnt_results.zip rnnt
!cd models && zip -qr {WORK}/rnnt_model.zip rnnt
if WORK == "/kaggle/working":
    print("Download rnnt_results.zip and rnnt_model.zip from the Output panel.")
else:
    from google.colab import files
    files.download(f"{WORK}/rnnt_results.zip")
    files.download(f"{WORK}/rnnt_model.zip")